# Lab 09 — Rejection Sampling → SFT

**Prerequisites:** Lecture 2 (RS section, rlhfbook.com/course) · Chapter 9 ([rlhfbook.com/c/09-rejection-sampling.html](https://rlhfbook.com/c/09-rejection-sampling.html)) · **your trained Lab 03 Bradley-Terry RM** (the scorer for this lab).

**Pipeline:** generate N=8 rollouts per GSM8K train prompt → score with your Lab 03 RM → select pairs via four arms (`top_per_prompt`, `random_per_prompt`, `top_k_overall`, `random_k_overall`) → SFT each → greedy exact-match eval on the test slice → **strategy-vs-matched-random-control table**.

> Scaffold notebook: cells define stage skeletons and TODOs. The answer key (`rlhf-book/code/rejection_sampling/`) opens only after your version trains.

In [ ]:
# Setup: imports + config load.
import json
from pathlib import Path

import starter  # lab scaffold: signatures, constants, fixtures

CONFIG_PATH = Path("configs/09_rejection_sampling_sft.yaml")
cfg = starter.load_config(str(CONFIG_PATH))  # validates structure

# TODO(Lab 09): point scorer.rm_checkpoint at your Lab 03 BT RM checkpoint,
# decide the policy model (Qwen3-0.6B Tier A vs 1.7B Tier B), and fix the seed.
print(cfg["selection"])

## Stage 1 — Rollout generation

N = `num_completions_per_prompt` completions per prompt, temperature/top-p per the config. **These parameters must stay identical across all four arms** so Stage 1+2 run once and land in a single cache file.

- [ ] generate rollouts; write `<hash>.jsonl` where the hash covers model + sampling params + seed + N + dataset slice
- [ ] record per-prompt rollout count so ragged rows are caught early (`starter.validate_rollout_records`)
- [ ] eyeball a few prompts → N completions before scoring anything

In [ ]:
# TODO(Lab 09): batch-generate N completions per prompt and build cache records:
# {"question": ..., "answer": ..., "completions": [...], "rewards": [...]}  # rewards filled in Stage 2
# raise NotImplementedError

## Stage 2 — Scoring with your Lab 03 RM

Bridge your BT reward model through `starter.RMScoreBridge.score`. Watch the two bugs you already debugged in Lab 03: pad side / pooling index. Optional second pass with `nvidia/AceMath-7B-RM` = production-RM comparison arm.

**Sanity-check before training** (methodology from the answer key's `diagnostics.py`, reimplement yourself):
1. reward histogram, correct vs incorrect completions
2. per-row winrate of argmax(reward) on *decidable* prompts vs random pick
3. best-of-N sweep (top-N-by-reward coverage vs random-N coverage)
4. `decidable_fraction` — how many prompts even allow within-row selection to matter? (data-side ceiling, not RM-side)

In [ ]:
# TODO(Lab 09): implement RMScoreBridge.score in starter.py, then score every
# completion and fill the rewards fields; persist one cache JSONL.
# raise NotImplementedError

## Stage 3a — Selection: four arms over ONE cache

| arm | keeps | control semantics |
|---|---|---|
| `top_per_prompt` | argmax-reward completion per prompt | classic RS |
| `random_per_prompt` | one uniform-random completion per prompt | matched size & coverage, seeded |
| `top_k_overall` | top-K over flat M×N matrix (prompt repetition allowed) | RM concentrates data |
| `random_k_overall` | K uniform-random pairs from the flat pool | matched budget, seeded |

`starter.run_selection(records, strategy, top_k=..., seed=...)` dispatches; implement the four stubs it calls.

In [ ]:
# TODO(Lab 09): implement select_* stubs in starter.py, then loop all four arms:
# for strategy in starter.STRATEGIES:
#     pairs = starter.run_selection(records, strategy, top_k=cfg["selection"]["top_k"], seed=cfg["seed"])
#     log len(pairs) to metrics JSONL — budgets must pair up (see deliverable table)
raise NotImplementedError("Stage 3a selection runs after implementing starter.select_*")

## Stage 3b — SFT each selected subset

Identical hyperparameters across arms (config §SFT block): lr 5e-6, 2 epochs, batch_acc etc. Append JSONL metrics per run (curriculum convention) so plots read from artifacts regardless of platform.

In [ ]:
# TODO(Lab 09): SFT loop (reuse your Lab 01 SFT trainer export if you like).
# One training run per arm; save final checkpoints under output_dir/<arm>/.
# raise NotImplementedError

## Stage 3c — Exact-match evaluation

Greedy decode (`eval_temperature: 0.0`) on the same fixed test slice for every arm. Implement `starter.extract_gsm8k_answer` + `starter.answers_match` + `starter.evaluate_exact_match`; report accuracy **and** unparsable-fraction separately.

In [ ]:
# TODO(Lab 09): eval every arm checkpoint on the identical test questions;
# collect {strategy -> test_accuracy} into results.json alongside n_pairs.
raise NotImplementedError("Stage 3c evaluation runs after implementing the exact-match stubs")

## Deliverable — strategy-vs-random table

Build with `starter.build_comparison_table(run_results)`: one row per arm with `paired_control`, `control_accuracy`, `delta_vs_control`, `budget_matched`.

**Motivation (read before running):** the answer key's published finding is that `top_k_overall` beat its matched random baseline while `top_per_prompt` tied `random_per_prompt` on its 1k-train/200-test slice. Candidate mechanisms to explain in writing after your table exists: overall top-K concentrates high-reward-and-correct pairs; per-prompt argmax is capped by the decidable fraction and takes exactly one possibly-wrong pick per prompt; small-slice noise can flip either ordering — which is precisely why matched controls exist.

Only now open the answer key: `rlhf-book/code/rejection_sampling/{selection.py, config.py, configs/*.yaml, diagnostics.py}` — all four configs + their reference runs on W&B.

In [ ]:
# TODO(Lab 09): load results.json, call starter.build_comparison_table, render rows
# (plain print or matplotlib — no pandas dependency required by this lab).
raise NotImplementedError("Comparison table builder pending implementation")